This file will generate columns called utterance_cossim_{category} that calculates the cosine similarity of a child's utterance to their six other utterances. It will also calculate the softmaxed probability by softmaxing over raw cosine similarity values to produce the columns utterance_probability_{category}. 

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax

In [2]:
EMBED_CSV = "../data/embeddings/utterance_half_and_full_embeddings.csv"
OUT_CSV = "../data/preprocessed/talk_cossim_toself.csv"

transcript_df = pd.read_csv(EMBED_CSV)

In [3]:
CATEGORIES = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]


In [4]:
def parse_embedding(x):
    if pd.isna(x) if not isinstance(x, np.ndarray) else False:
        return np.nan
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, str):
        return np.fromstring(x.strip("[]"), sep=" ")
    return np.nan

In [5]:
def add_within_subject_utterance_similarities(
    transcript_df,
    embedding_col="utterance_full_embedding",
    participant_col="participant_id",
    category_col="target_category",
    categories=CATEGORIES
):
    transcript_df = transcript_df.copy()
    transcript_df[embedding_col] = transcript_df[embedding_col].apply(parse_embedding)
    transcript_df[category_col] = transcript_df[category_col].str.lower().str.strip()

    for cat in categories:
        transcript_df[f"utterance_cossim_{cat}"] = np.nan
        transcript_df[f"utterance_probability_{cat}"] = np.nan

    for pid, g in transcript_df.groupby(participant_col):
        # Build per-category mean embedding, skipping NaN rows
        cat_to_emb = {}
        for cat in categories:
            embs = [
                e for e in g.loc[g[category_col] == cat, embedding_col]
                if isinstance(e, np.ndarray) and e.ndim == 1
            ]
            if embs:
                cat_to_emb[cat] = np.mean(np.stack(embs), axis=0)

        for idx, row in g.iterrows():
            query_emb = row[embedding_col]
            if not isinstance(query_emb, np.ndarray) or query_emb.ndim != 1:
                continue

            # Exclude the current row's own category (matches 04c logic)
            row_cat = row[category_col]
            comparison_cats = [c for c in categories if c != row_cat and c in cat_to_emb]
            if not comparison_cats:
                continue

            category_matrix = np.stack([cat_to_emb[c] for c in comparison_cats])
            sims = cosine_similarity(query_emb.reshape(1, -1), category_matrix).flatten()
            probs = softmax(sims)

            for cat, sim, prob in zip(comparison_cats, sims, probs):
                transcript_df.loc[idx, f"utterance_cossim_{cat}"] = sim
                transcript_df.loc[idx, f"utterance_probability_{cat}"] = prob

    return transcript_df

In [6]:
transcript_df.head()

,participant_id,target_category,utterance_masked,utterance_half_masked_filtered,utterance_full_embedding,utterance_half_embedding,age_months
0,AD001,crab,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...",[-3.59437495e-01 1.92119449e-01 2.10224107e-...,[-9.4677097e-01 1.9568498e-01 -9.0820402e-01 ...,229.0
1,AD001,whale,Um it looks like a big creature and it has a l...,it looks like a big creature and it has a lot ...,[ 2.242232 0.6355182 -0.41715586 0.976658...,[ 1.61588228e+00 1.05946887e+00 -7.86195457e-...,229.0
2,AD001,turtle,It has a hard shell and four legs which helps ...,It has a hard shell and four legs which helps ...,[-0.60391587 1.9758062 0.03733304 0.992889...,[ 4.81736600e-01 8.82052124e-01 -1.17873788e-...,229.0
3,AD001,shark,It's a big sea creature and it's usually grey....,It's a big sea creature and it's usually grey....,[ 5.61921597e-01 3.19676846e-01 -2.11030498e-...,[ 8.51729989e-01 -7.92751089e-02 -4.88009453e-...,229.0
4,AD001,seahorse,It looks small and it's usually like a bright ...,It looks small and it's usually like a bright ...,[ 7.55800068e-01 6.24026775e-01 1.78464040e-...,[-9.37828347e-02 -4.86495882e-01 7.92466104e-...,229.0


In [7]:
transcript_df = add_within_subject_utterance_similarities(transcript_df)

In [8]:
cols = [f"utterance_probability_{c}" for c in CATEGORIES]
transcript_df["prob_sum"] = transcript_df[cols].sum(axis=1)
transcript_df["prob_sum"].describe()

count    647.000000
mean       0.839258
std        0.367577
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: prob_sum, dtype: float64

In [9]:
transcript_df.head(10)

,participant_id,target_category,utterance_masked,utterance_half_masked_filtered,utterance_full_embedding,utterance_half_embedding,age_months,utterance_cossim_crab,utterance_probability_crab,utterance_cossim_octopus,utterance_probability_octopus,utterance_cossim_seahorse,utterance_probability_seahorse,utterance_cossim_shark,utterance_probability_shark,utterance_cossim_turtle,utterance_probability_turtle,utterance_cossim_whale,utterance_probability_whale,prob_sum
0,AD001,crab,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...","[-0.359437495, 0.192119449, 0.210224107, 0.687...",[-9.4677097e-01 1.9568498e-01 -9.0820402e-01 ...,229.0,NaN,NaN,0.433996,0.241211,0.381785,0.228940,0.038519,0.162421,0.032749,0.161487,0.275912,0.205940,1.0
1,AD001,whale,Um it looks like a big creature and it has a l...,it looks like a big creature and it has a lot ...,"[2.242232, 0.6355182, -0.41715586, 0.9766586, ...",[ 1.61588228e+00 1.05946887e+00 -7.86195457e-...,229.0,0.275912,0.185769,0.301351,0.190555,0.584005,0.252800,0.454114,0.222006,0.054480,0.148870,NaN,NaN,1.0
2,AD001,turtle,It has a hard shell and four legs which helps ...,It has a hard shell and four legs which helps ...,"[-0.60391587, 1.9758062, 0.03733304, 0.9928895...",[ 4.81736600e-01 8.82052124e-01 -1.17873788e-...,229.0,0.032749,0.192246,0.103224,0.206284,0.028379,0.191408,0.138040,0.213592,NaN,NaN,0.054480,0.196470,1.0
3,AD001,shark,It's a big sea creature and it's usually grey....,It's a big sea creature and it's usually grey....,"[0.561921597, 0.319676846, -0.211030498, -0.66...",[ 8.51729989e-01 -7.92751089e-02 -4.88009453e-...,229.0,0.038519,0.170499,-0.001411,0.163825,0.288786,0.218983,NaN,NaN,0.138040,0.188340,0.454114,0.258352,1.0
4,AD001,seahorse,It looks small and it's usually like a bright ...,It looks small and it's usually like a bright ...,"[0.755800068, 0.624026775, 0.17846404, 0.93242...",[-9.37828347e-02 -4.86495882e-01 7.92466104e-...,229.0,0.381785,0.204764,0.426828,0.214198,NaN,NaN,0.288786,0.186580,0.028379,0.143804,0.584005,0.250655,1.0
5,AD001,octopus,It has eight legs and it's usually also bright...,It has eight legs and it's usually also bright...,"[0.629135191, -0.0630739704, 1.44510567, 1.614...",[ 4.97932523e-01 -5.16983628e-01 1.08119667e+...,229.0,0.433996,0.236183,NaN,NaN,0.426828,0.234496,-0.001411,0.152811,0.103224,0.169667,0.301351,0.206843,1.0
6,AD002,crab,"Also, it can be all sorts of colors and those ...","Also, it can be all sorts of colors and those ...","[-0.74769914, 0.34029394, 0.05867911, 1.420595...",[-0.975324 1.063338 -0.23174666 1.649707...,337.0,NaN,NaN,0.093313,0.247452,-0.038517,0.216889,NaN,NaN,0.304719,0.305706,0.019966,0.229952,1.0
7,AD002,octopus,They have um eight tentacles coming from their...,They have eight tentacles coming from their ro...,"[-0.9793499, 1.7595148, 0.6887004, -0.45271325...",[-1.2820868 1.2046236 0.50795484 -0.208396...,337.0,0.093313,0.266404,NaN,NaN,0.076842,0.262052,NaN,NaN,-0.035930,0.234106,-0.021799,0.237437,1.0
8,AD002,shark,NaN,NaN,NaN,NaN,337.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
9,AD002,turtle,"Okay. It has, is a shell that is on the bottom...","Okay. It has, is a shell that is on the bottom...","[-2.25198483, 1.40264964, -1.78706825, 1.57430...",[-2.33796096e+00 1.76335716e+00 -1.39189035e-...,337.0,0.304719,0.315243,-0.035930,0.224235,-0.015443,0.228876,NaN,NaN,NaN,NaN,-0.003411,0.231647,1.0


In [10]:
cols_to_drop = [
    "utterance_masked", "utterance_half_masked_filtered",
    "utterance_full_embedding", "utterance_half_embedding", "prob_sum"
]
result = transcript_df.drop(columns=[c for c in cols_to_drop if c in transcript_df.columns])

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"Saved {result.shape[0]} rows → {OUT_CSV}")
print(result.columns.tolist())

Saved 647 rows → ../data/preprocessed/talk_cossim_toself.csv
['participant_id', 'target_category', 'age_months', 'utterance_cossim_crab', 'utterance_probability_crab', 'utterance_cossim_octopus', 'utterance_probability_octopus', 'utterance_cossim_seahorse', 'utterance_probability_seahorse', 'utterance_cossim_shark', 'utterance_probability_shark', 'utterance_cossim_turtle', 'utterance_probability_turtle', 'utterance_cossim_whale', 'utterance_probability_whale']
